# Echoes of Arcana — Google Colab Training (T4 GPU / CPU → `spell_model.onnx`)

Обучение компактной 2D-CNN на 4 классах (`background`, `virdan`, `noktar`, `targum`) и экспорт в оффлайн-модель **`spell_model.onnx`** для Godot 4.

По умолчанию автоматически скачивает `bootstrap_dataset.zip` из `Alizhan2/Game-x`. Когда команда запишет свои голоса через `web_recorder.html`, переключите `UPLOAD_CUSTOM_ZIP = True`.

In [ ]:
# Ячейка 1: Проверка GPU и установка зависимостей (включая onnxscript)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

!pip -q install onnx onnxscript onnxruntime scikit-learn matplotlib soundfile

In [ ]:
# Ячейка 2: Получение датасета (автоскачивание из GitHub или своя загрузка)
UPLOAD_CUSTOM_ZIP = False
BOOTSTRAP_URL = "https://raw.githubusercontent.com/Alizhan2/Game-x/main/ai_voice/bootstrap_dataset.zip"

import urllib.request
import zipfile
from pathlib import Path

DATA_DIR = Path("dataset_raw")
if UPLOAD_CUSTOM_ZIP:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".zip"):
            with zipfile.ZipFile(fname, "r") as zf:
                zf.extractall(".")
            print(f"Распакован пользовательский архив: {fname}")
else:
    zip_path = Path("bootstrap_dataset.zip")
    print("Скачиваю стартовый датасет из", BOOTSTRAP_URL)
    urllib.request.urlretrieve(BOOTSTRAP_URL, str(zip_path))
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(".")
    print("Распаковано в", DATA_DIR.resolve())

In [ ]:
# Ячейка 3: Стратифицированное разделение ДО аугментации + 32x32 log-Mel
import math
import random
import numpy as np
import soundfile as sf
from sklearn.model_selection import train_test_split

TARGET_SR = 16000
CLIP_SECONDS = 1.2
TARGET_SAMPLES = int(TARGET_SR * CLIP_SECONDS)
LABELS = ["background", "virdan", "noktar", "targum"]

def resample_to_16k(samples, orig_sr):
    src = np.asarray(samples, dtype=np.float32)
    if src.ndim > 1:
        src = src.mean(axis=1)
    if orig_sr == TARGET_SR or len(src) == 0:
        return src
    out_len = max(1, int(round(len(src) * (TARGET_SR / float(orig_sr)))))
    x_old = np.linspace(0.0, 1.0, len(src))
    x_new = np.linspace(0.0, 1.0, out_len)
    return np.interp(x_new, x_old, src).astype(np.float32)

def pad_or_trim(arr, target_len=TARGET_SAMPLES):
    if len(arr) >= target_len:
        return arr[:target_len]
    return np.pad(arr, (0, target_len - len(arr)))

def augment_wave(samples, rng):
    n = len(samples)
    shift = rng.randint(-int(0.08 * n), int(0.08 * n))
    gain = rng.uniform(0.55, 1.35)
    noise_amp = rng.uniform(0.002, 0.025)
    rolled = np.roll(samples, shift) * gain
    noise = np.random.uniform(-noise_amp, noise_amp, size=n).astype(np.float32)
    return np.clip(rolled + noise, -1.0, 1.0)

def extract_log_mel(samples, sr=TARGET_SR, n_mels=32, n_frames=32):
    wave = pad_or_trim(resample_to_16k(samples, sr), TARGET_SAMPLES)
    n_fft = 512
    hop = max(1, (len(wave) - n_fft) // max(1, n_frames - 1))
    window = np.hanning(n_fft).astype(np.float32)
    frames = []
    for i in range(n_frames):
        start = i * hop
        chunk = wave[start : start + n_fft]
        if len(chunk) < n_fft:
            chunk = np.pad(chunk, (0, n_fft - len(chunk)))
        spec = np.abs(np.fft.rfft(chunk * window)) ** 2
        frames.append(spec)
    power_spec = np.stack(frames, axis=1)
    mel_min = 2595.0 * math.log10(1.0 + 60.0 / 700.0)
    mel_max = 2595.0 * math.log10(1.0 + (TARGET_SR / 2.0) / 700.0)
    mel_pts = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_pts = 700.0 * (10.0 ** (mel_pts / 2595.0) - 1.0)
    bins = np.floor((n_fft + 1) * hz_pts / TARGET_SR).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for m in range(1, n_mels + 1):
        l, c, r = bins[m - 1], bins[m], bins[m + 1]
        for k in range(l, c):
            if c > l: fb[m - 1, k] = (k - l) / float(c - l)
        for k in range(c, r):
            if r > c: fb[m - 1, k] = (r - k) / float(r - c)
    log_mel = np.log(np.maximum(1e-6, fb @ power_spec))
    std = float(log_mel.std())
    norm = (log_mel - float(log_mel.mean())) / (std if std > 1e-5 else 1.0)
    return norm.astype(np.float32)[None, :, :]  # (1, 32, 32)

items = []
for idx, label in enumerate(LABELS):
    for wav_path in sorted((DATA_DIR / label).glob("*.wav")):
        items.append((wav_path, idx))

train_items, val_items = train_test_split(
    items, test_size=0.25, random_state=42, stratify=[y for _, y in items]
)

rng = random.Random(42)
np.random.seed(42)
x_tr, y_tr = [], []
for wav_path, idx in train_items:
    data, sr = sf.read(str(wav_path), dtype="float32")
    mono_16k = resample_to_16k(data, sr)
    x_tr.append(extract_log_mel(mono_16k, TARGET_SR))
    y_tr.append(idx)
    for _ in range(10):
        x_tr.append(extract_log_mel(augment_wave(mono_16k, rng), TARGET_SR))
        y_tr.append(idx)

x_val, y_val = [], []
x_val_raw, y_val_raw = [], []
for wav_path, idx in val_items:
    data, sr = sf.read(str(wav_path), dtype="float32")
    mono_16k = resample_to_16k(data, sr)
    feat = extract_log_mel(mono_16k, TARGET_SR)
    x_val_raw.append(feat)
    y_val_raw.append(idx)
    x_val.append(feat)
    y_val.append(idx)
    for _ in range(2):
        x_val.append(extract_log_mel(augment_wave(mono_16k, rng), TARGET_SR))
        y_val.append(idx)

X_tr = np.stack(x_tr).astype(np.float32)
Y_tr = np.asarray(y_tr, dtype=np.int64)
X_val = np.stack(x_val).astype(np.float32)
Y_val = np.asarray(y_val, dtype=np.int64)
X_val_raw = np.stack(x_val_raw).astype(np.float32)
Y_val_raw = np.asarray(y_val_raw, dtype=np.int64)
print(f"Train: {X_tr.shape} | Val (all): {X_val.shape} | Val (raw unseen): {X_val_raw.shape}")

In [ ]:
# Ячейка 4: Mini-batch обучение SpellCNN + сохранение метрик и графиков для жюри
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

class SpellCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 48, kernel_size=3, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.dropout = nn.Dropout(0.15)
        self.classifier = nn.Linear(48, num_classes)

    def forward(self, x):
        feat = self.dropout(self.features(x).flatten(1))
        return self.classifier(feat)

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SpellCNN(len(LABELS)).to(device)
EPOCHS = 45
opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(Y_tr)),
    batch_size=32,
    shuffle=True,
)
X_val_t, Y_val_t = torch.from_numpy(X_val).to(device), torch.from_numpy(Y_val).to(device)
X_raw_t, Y_raw_t = torch.from_numpy(X_val_raw).to(device), torch.from_numpy(Y_val_raw).to(device)

history_loss, history_acc = [], []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        opt.zero_grad()
        loss = loss_fn(model(bx), by)
        loss.backward()
        opt.step()
        total_loss += float(loss.item()) * len(bx)
    scheduler.step()
    avg_loss = total_loss / max(1, len(X_tr))

    model.eval()
    with torch.no_grad():
        preds = model(X_val_t).argmax(dim=1)
        acc = float((preds == Y_val_t).float().mean()) * 100.0
        raw_preds = model(X_raw_t).argmax(dim=1)
        raw_acc = float((raw_preds == Y_raw_t).float().mean()) * 100.0
    history_loss.append(avg_loss)
    history_acc.append(acc)
    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={avg_loss:.4f} | val_acc={acc:.1f}% | raw_val_acc={raw_acc:.1f}%")

model.eval()
with torch.no_grad():
    val_preds = model(X_val_t).argmax(dim=1).cpu().numpy()
    raw_val_preds = model(X_raw_t).argmax(dim=1).cpu().numpy()

raw_final = round(float((raw_val_preds == Y_val_raw).mean()) * 100.0, 2)
metrics = {
    "final_val_acc": round(history_acc[-1], 2),
    "raw_unseen_val_acc": raw_final,
    "final_loss": round(history_loss[-1], 4),
    "train_samples": int(len(X_tr)),
    "val_samples": int(len(X_val)),
    "labels": LABELS,
}
Path("training_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"FINAL_VAL_ACC={metrics['final_val_acc']}% | RAW_UNSEEN_VAL_ACC={raw_final}%")
print(classification_report(Y_val, val_preds, target_names=LABELS))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_acc, label="Val Accuracy (%)", color="#38bdf8")
axes[0].set_title("SpellCNN Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].grid(True, alpha=0.3)
cm = confusion_matrix(Y_val, val_preds)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Ячейка 5: Экспорт в spell_model.onnx + проверка onnxruntime + скачивание
import inspect
from pathlib import Path
import onnxruntime as ort

model_cpu = model.eval().cpu()
dummy = torch.randn(1, 1, 32, 32)
export_kwargs = {
    "input_names": ["mel"],
    "output_names": ["logits"],
    "dynamic_axes": {"mel": {0: "batch"}, "logits": {0: "batch"}},
    "opset_version": 17,
}
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    export_kwargs["dynamo"] = False

torch.onnx.export(model_cpu, dummy, "spell_model.onnx", **export_kwargs)

sess = ort.InferenceSession("spell_model.onnx", providers=["CPUExecutionProvider"])
test_out = sess.run(["logits"], {"mel": X_val_raw[:1]})[0]
size_kb = Path("spell_model.onnx").stat().st_size / 1024.0
cm_ok = Path("confusion_matrix.png").is_file()
print(f"ONNX_EXPORT_SUCCESS | size={size_kb:.1f} KB | confusion_matrix={cm_ok}")
print("Проверка инференса: предсказано=", LABELS[int(test_out.argmax())], "| истинный=", LABELS[int(Y_val_raw[0])])

try:
    from google.colab import files
    files.download("spell_model.onnx")
    files.download("confusion_matrix.png")
except Exception as exc:
    print("Авто-скачивание пропущено (скачай из панели Files слева):", exc)